In [24]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
import shutil

In [25]:
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [26]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"}
)

In [27]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [28]:
vector_store = FAISS.from_documents(docs, embedding)

In [30]:
vector_store.save_local("faiss_index")

In [31]:
# vector_store = Chroma(
#     embedding_function=embedding,
#     persist_directory='my_chroma_db',
#     collection_name='sample'
# )

vector_store = FAISS.load_local(
    "faiss_index",
    embeddings=embedding,
    allow_dangerous_deserialization=True
)

In [16]:
# vector_store.delete_collection()

shutil.rmtree("faiss_index", ignore_errors=True)

In [ ]:
# add documents
# vector_store.add_documents(docs)

['db6e0c80-0f37-497c-8279-e0141cb8283e',
 'a5bc958b-5ee2-4108-b116-b48d5b23d9ab',
 'f494061c-2948-45f0-8df7-c7b567c05456',
 '8d78ec56-44d0-48b3-b722-85d4200c0417',
 'bb66bf4f-de15-4f70-a078-6295fbc496f5']

In [32]:
# view documents
# vector_store.get(include=['embeddings','documents', 'metadatas'])

docs_dict = vector_store.docstore._dict

for doc_id, doc in docs_dict.items():
    print(doc_id, doc.page_content)


95a1725f-77dd-4704-8b01-164b73994477 Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.
cd61571b-ef4b-4de3-9c3a-b2ae99decbeb Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.
3dde0533-735e-47fa-bae5-238889168380 MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.
17853079-05f5-491e-bb04-75ba823b5b72 Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.
9d10621a-9de7-4c6b-8c0b-93d3693dacc0 Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his qu

In [33]:
# search documents
# vector_store.similarity_search(
#     query='Who among these are a bowler?',
#     k=2
# )

docs = vector_store.similarity_search("Who are batsman among them..?", k=3)

In [34]:
for doc in docs:
    print(doc.page_content)

Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.
MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.
Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.


In [76]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who is the best batsman among them?',
    k=2
)

[(Document(metadata={'team': 'Royal Challengers Bangalore'}, page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket."),
  0.8942601680755615),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.9723703265190125)]

In [65]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8879573345184326),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  2.0017192363739014)]

In [68]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='41b562b6-f2d3-4f62-aced-611e85139e27', document=updated_doc1)


In [71]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['41b562b6-f2d3-4f62-aced-611e85139e27',
  'bb4da8b2-d984-48f2-abff-26542fe4296a',
  '0b7143c5-2b7b-48af-8685-b47d9e54db60',
  'b38c0cb0-f209-40a5-a02c-94fe24f3a213',
  '683b81a0-9f79-4109-81c3-2a89c12943c0'],
 'embeddings': array([[-0.04108337, -0.02528015, -0.0214507 , ...,  0.02231625,
         -0.02287274, -0.00913005],
        [-0.01992594,  0.00161873, -0.00959343, ...,  0.02433111,
         -0.00341398, -0.01722028],
        [-0.04256179,  0.0106033 , -0.01305768, ...,  0.04738369,
         -0.01748699,  0.00204661],
        [-0.02734397, -0.0330087 ,  0.01565377, ...,  0.01462703,
          0.01177979,  0.00556418],
        [ 0.02301689, -0.02011725, -0.00123582, ...,  0.02533475,
          0.00280528, -0.05120279]], shape=(5, 768)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple ce

In [72]:
# delete document
vector_store.delete(ids=['b38c0cb0-f209-40a5-a02c-94fe24f3a213'])

In [73]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['41b562b6-f2d3-4f62-aced-611e85139e27',
  'bb4da8b2-d984-48f2-abff-26542fe4296a',
  '0b7143c5-2b7b-48af-8685-b47d9e54db60',
  '683b81a0-9f79-4109-81c3-2a89c12943c0'],
 'embeddings': array([[-0.04108337, -0.02528015, -0.0214507 , ...,  0.02231625,
         -0.02287274, -0.00913005],
        [-0.01992594,  0.00161873, -0.00959343, ...,  0.02433111,
         -0.00341398, -0.01722028],
        [-0.04256179,  0.0106033 , -0.01305768, ...,  0.04738369,
         -0.01748699,  0.00204661],
        [ 0.02301689, -0.02011725, -0.00123582, ...,  0.02533475,
          0.00280528, -0.05120279]], shape=(4, 768)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league